# Standalone baseline models

Run after **feature extraction** and **data splitting**, and before **model_development.ipynb**. This notebook trains the three baselines the paper compares the hybrid model against, all as XGBoost classifiers:

| Baseline | Input | Where it comes from |
|----------|-------|---------------------|
| **spaCy-only** | **S**, the 10 stylometric features | `spacy_features.parquet` |
| **ELECTRA-only** | **E**, the 768-dim CLS embedding | `electra_embeddings.npy` |
| **XGBoost-only** | **R**, TF-IDF of the preprocessed text (no spaCy, no ELECTRA) | built here, vocabulary learned on train rows only |

The conditions match the hybrid exactly (Chapter 3, §7): the same 70/15/15 row picks from `data/processed/splits/`, the same grid, the same grouped 5-fold cross-validation on the 70% train rows with the three best settings refit on all of them, and the same validation rule to choose between them.

This notebook trains and selects only. It never scores the test slice: Table 6, the per-model FPR, and the paired t-test are produced once, for all four models together, in `model_development.ipynb`, which loads the baselines saved here. Run this notebook first.

Outputs land in `data/processed/models/`. The hybrid model's own files are not touched.

In [1]:
import os

# Hugging Face download bars are widgets that VS Code can't render; turn them off before transformers loads.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import sys
from pathlib import Path

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Re-run this cell after editing src/utils so Jupyter does not keep old modules.
for _name in list(sys.modules):
    if _name == "utils" or _name.startswith("utils."):
        del sys.modules[_name]

DATA_DIR = PROJECT_ROOT / "data"
COMBINED_CSV = DATA_DIR / "processed" / "combined_dataset.csv"
FEATURES_DIR = DATA_DIR / "processed" / "features"
SPLITS_DIR = DATA_DIR / "processed" / "splits"
MODELS_DIR = DATA_DIR / "processed" / "models"
SPACY_PARQUET = FEATURES_DIR / "spacy_features.parquet"
ELECTRA_NPY = FEATURES_DIR / "electra_embeddings.npy"

RANDOM_STATE = 42
CV_FOLDS = 5
USE_CUDA = True  # set False for a CPU smoke run; the full grid belongs on the GPU

print("PROJECT_ROOT:", PROJECT_ROOT)
for path in (COMBINED_CSV, SPACY_PARQUET, ELECTRA_NPY, SPLITS_DIR / "split_assignments.csv"):
    print(f"{path.name} exists:", path.exists())

PROJECT_ROOT: /workspace/hybrid-ai-framework
combined_dataset.csv exists: True
spacy_features.parquet exists: True
electra_embeddings.npy exists: True
split_assignments.csv exists: True


**GPU SMOKE TEST**

In [2]:
import torch
from utils.features.split_and_train import smoke_test_xgboost_cuda

if USE_CUDA:
    print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
    smoke_test_xgboost_cuda(RANDOM_STATE)
    print("XGBoost GPU smoke test OK")
else:
    print("USE_CUDA=False, skipping the GPU smoke test.")

CUDA: True NVIDIA GeForce RTX 4090
XGBoost GPU smoke test OK


## Load labels, splits, **S** and **E**

Row order must match `combined_dataset.csv`. Every baseline sees the same train rows, validation rows, and test rows as the hybrid. Group ids come from the split file, where each MGTBench polished text already shares its original's id, so cross-validation folds never separate related rows.

In [3]:
from utils.features.data_splitting import load_split_indices
from utils.features.spacy_features import SPACY_FEATURE_NAMES

for path in (COMBINED_CSV, SPACY_PARQUET, ELECTRA_NPY):
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}: run data_cleaning, data_splitting and feature_extraction first.")

df = pd.read_csv(COMBINED_CSV)
y = df["label"].to_numpy(dtype=np.int64)
sources = df["source"].to_numpy()
n_rows = len(df)

idx_train, idx_val, idx_test = load_split_indices(SPLITS_DIR)
all_idx = np.concatenate([idx_train, idx_val, idx_test])
if len(all_idx) != n_rows or len(np.unique(all_idx)) != n_rows:
    raise ValueError("Split indices must be disjoint and cover every row of the combined table.")

assignments = pd.read_csv(SPLITS_DIR / "split_assignments.csv")
if len(assignments) != n_rows or not (assignments["row_id"].to_numpy() == np.arange(n_rows)).all():
    raise ValueError("split_assignments.csv does not match the combined table; re-run data_splitting.ipynb.")
if not (assignments["label"].to_numpy() == y).all():
    raise ValueError("split_assignments.csv labels differ from the combined table; re-run data_splitting.ipynb.")
groups = assignments["group_id"].to_numpy()

# S: the ten spaCy stylometric features, in the order of SPACY_FEATURE_NAMES.
spacy_frame = pd.read_parquet(SPACY_PARQUET)
if len(spacy_frame) != n_rows or not (spacy_frame["label"].to_numpy() == y).all():
    raise ValueError("spacy_features.parquet does not match the combined table; re-run feature_extraction.ipynb.")
S = spacy_frame[SPACY_FEATURE_NAMES].to_numpy(dtype=np.float32)

# E: the 768-dim ELECTRA CLS embeddings.
E = np.load(ELECTRA_NPY)
if E.shape[0] != n_rows:
    raise ValueError(f"E has {E.shape[0]} rows, combined table has {n_rows}")

print("Rows:", n_rows, "| S:", S.shape, "| E:", E.shape)
print("Train / val / test:", len(idx_train), len(idx_val), len(idx_test))
print("Human share in train:", f"{(y[idx_train] == 0).mean():.1%}")

Rows: 299659 | S: (299659, 10) | E: (299659, 768)
Train / val / test: 209535 44976 45148
Human share in train: 48.3%


## **R** for the XGBoost-only baseline

TF-IDF over the same placeholder-stripped text that spaCy and ELECTRA read. The vocabulary and word weights are learned from **train rows only**, so nothing from validation or test leaks in. The matrix stays sparse.

In [4]:
import joblib

from utils.features.split_and_train import fit_tfidf_on_train
from utils.text_prep import strip_placeholders

texts = np.array([strip_placeholders(t) for t in df["text"]], dtype=object)
vectorizer = fit_tfidf_on_train(texts[idx_train])
R = vectorizer.transform(texts)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(vectorizer, MODELS_DIR / "tfidf_vectorizer.joblib")
print("R:", R.shape, "| non-zeros:", R.nnz, "| vocabulary:", len(vectorizer.vocabulary_))

R: (299659, 10000) | non-zeros: 42086283 | vocabulary: 10000


## Training (70%): grid search + 5-fold CV

Each baseline goes through `train_xgboost_gpu`, the same function and the same grid as the hybrid: grouped 5-fold cross-validation on the train rows, scored by **ROC-AUC minus FPR**, then the best setting refit on all train rows. Validation and test rows are not used here.

The full grid is 32 settings × 5 folds plus a refit, for each of the three models, so this cell takes a while.

In [5]:
import json

from utils.features.split_and_train import train_xgboost_gpu

FEATURES = {"spacy": S, "electra": E, "tfidf": R}
LABELS = {"spacy": "spaCy-only", "electra": "ELECTRA-only", "tfidf": "XGBoost-only"}

searches, cv_results = {}, {}
for name, X in FEATURES.items():
    print(f"\n=== {LABELS[name]} ({X.shape[1]} features) ===")
    _, search = train_xgboost_gpu(
        X[idx_train],
        y[idx_train],
        groups=groups[idx_train],
        cv_folds=CV_FOLDS,
        random_state=RANDOM_STATE,
        use_cuda=USE_CUDA,
    )
    best = search.best_index_
    cv_mean = float(search.cv_results_["mean_test_score"][best])
    cv_std = float(search.cv_results_["std_test_score"][best])
    print("Best params:", search.best_params_)
    print(f"Best CV score (ROC-AUC - FPR): {cv_mean:.4f} +/- {cv_std:.4f} over {CV_FOLDS} folds")

    rank_cols = [c for c in search.cv_results_ if c.startswith("param_")] + ["mean_test_score", "std_test_score"]
    cv_rank = pd.DataFrame(search.cv_results_)[rank_cols].sort_values("mean_test_score", ascending=False)
    display(cv_rank.head(5))

    searches[name] = search
    cv_results[name] = {
        "cv_folds": CV_FOLDS,
        "n_features": int(X.shape[1]),
        "cv_best_params": search.best_params_,
        "cv_mean_score_auc_minus_fpr": cv_mean,
        "cv_std_score_auc_minus_fpr": cv_std,
        "top_configs": cv_rank.head(8).to_dict(orient="records"),
    }


=== spaCy-only (10 features) ===
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best params: {'gamma': 0.0, 'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 50, 'n_estimators': 400, 'reg_lambda': 10.0, 'subsample': 0.8}
Best CV score (ROC-AUC - FPR): 0.4697 +/- 0.0025 over 5 folds


,param_gamma,param_learning_rate,param_max_depth,param_min_child_weight,param_n_estimators,param_reg_lambda,param_subsample,mean_test_score,std_test_score
15,0.0,0.1,2,50,400,10.0,0.8,0.469723,0.002538
31,1.0,0.1,2,50,400,10.0,0.8,0.469723,0.002538
29,1.0,0.1,2,10,400,10.0,0.8,0.469514,0.002705
12,0.0,0.1,2,10,400,1.0,0.8,0.469483,0.003089
28,1.0,0.1,2,10,400,1.0,0.8,0.469483,0.003089



=== ELECTRA-only (768 features) ===
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best params: {'gamma': 0.0, 'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 10, 'n_estimators': 400, 'reg_lambda': 1.0, 'subsample': 0.8}
Best CV score (ROC-AUC - FPR): 0.8675 +/- 0.0034 over 5 folds


,param_gamma,param_learning_rate,param_max_depth,param_min_child_weight,param_n_estimators,param_reg_lambda,param_subsample,mean_test_score,std_test_score
12,0.0,0.1,2,10,400,1.0,0.8,0.867457,0.003395
28,1.0,0.1,2,10,400,1.0,0.8,0.867457,0.003395
30,1.0,0.1,2,50,400,1.0,0.8,0.867130,0.003215
14,0.0,0.1,2,50,400,1.0,0.8,0.867130,0.003215
13,0.0,0.1,2,10,400,10.0,0.8,0.866483,0.003113



=== XGBoost-only (10000 features) ===
Fitting 5 folds for each of 32 candidates, totalling 160 fits


/workspace/hybrid-ai-framework/venv/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [23:31:15] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


Best params: {'gamma': 1.0, 'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 10, 'n_estimators': 400, 'reg_lambda': 1.0, 'subsample': 0.8}
Best CV score (ROC-AUC - FPR): 0.8153 +/- 0.0022 over 5 folds


,param_gamma,param_learning_rate,param_max_depth,param_min_child_weight,param_n_estimators,param_reg_lambda,param_subsample,mean_test_score,std_test_score
28,1.0,0.1,2,10,400,1.0,0.8,0.815344,0.002225
30,1.0,0.1,2,50,400,1.0,0.8,0.815343,0.002692
12,0.0,0.1,2,10,400,1.0,0.8,0.815101,0.002462
14,0.0,0.1,2,50,400,1.0,0.8,0.814528,0.003222
15,0.0,0.1,2,50,400,10.0,0.8,0.813372,0.003363


## Validation (15%): choose between the trained models

The three best settings from each grid search are each fit on all train rows. The validation slice compares them on the same **ROC-AUC minus FPR** score and the best one becomes that baseline's final model. Validation rows only choose between these candidates: nothing is tuned or refit on them, and no final metrics are reported here. The test slice is scored once, in `model_development.ipynb`.

In [6]:
from utils.features.split_and_train import select_on_validation

TOP_K = 3

models, selection_tables = {}, []
for name, X in FEATURES.items():
    print(f"\n=== {LABELS[name]} ===")
    model, params, table = select_on_validation(
        searches[name],
        X[idx_train],
        y[idx_train],
        X[idx_val],
        y[idx_val],
        top_k=TOP_K,
        random_state=RANDOM_STATE,
        use_cuda=USE_CUDA,
    )
    display(table)
    models[name] = model
    selection_tables.append(table.assign(model=LABELS[name]))
    cv_results[name]["selected_params"] = params
    model.save_model(str(MODELS_DIR / f"xgb_{name}.json"))

pd.concat(selection_tables, ignore_index=True).to_csv(MODELS_DIR / "baseline_validation_selection.csv", index=False)
(MODELS_DIR / "baseline_cv_results.json").write_text(
    json.dumps(cv_results, indent=2, default=str), encoding="utf-8"
)
print(
    "\nSaved xgb_spacy.json, xgb_electra.json, xgb_tfidf.json, tfidf_vectorizer.joblib, "
    "baseline_validation_selection.csv and baseline_cv_results.json to", MODELS_DIR
)


=== spaCy-only ===
Validation picked CV rank 1 of 3: score 0.4730, FPR 32.159%


,cv_rank,param_gamma,param_learning_rate,param_max_depth,param_min_child_weight,param_n_estimators,param_reg_lambda,param_subsample,cv_mean,cv_std,val_roc_auc,val_fpr,val_score,selected
0,1,0.0,0.1,2,50,400,10.0,0.8,0.469723,0.002538,0.794552,0.321592,0.472960,True
1,2,1.0,0.1,2,50,400,10.0,0.8,0.469723,0.002538,0.794410,0.321638,0.472772,False
2,3,1.0,0.1,2,10,400,10.0,0.8,0.469514,0.002705,0.794484,0.321684,0.472800,False



=== ELECTRA-only ===
Validation picked CV rank 3 of 3: score 0.8668, FPR 10.096%


,cv_rank,param_gamma,param_learning_rate,param_max_depth,param_min_child_weight,param_n_estimators,param_reg_lambda,param_subsample,cv_mean,cv_std,val_roc_auc,val_fpr,val_score,selected
0,1,0.0,0.1,2,10,400,1.0,0.8,0.867457,0.003395,0.967806,0.101057,0.866749,False
1,2,1.0,0.1,2,10,400,1.0,0.8,0.867457,0.003395,0.967806,0.101057,0.866749,False
2,3,0.0,0.1,2,50,400,1.0,0.8,0.867130,0.003215,0.967799,0.100965,0.866834,True



=== XGBoost-only ===
Validation picked CV rank 2 of 3: score 0.8199, FPR 11.112%


,cv_rank,param_gamma,param_learning_rate,param_max_depth,param_min_child_weight,param_n_estimators,param_reg_lambda,param_subsample,cv_mean,cv_std,val_roc_auc,val_fpr,val_score,selected
0,1,1.0,0.1,2,10,400,1.0,0.8,0.815344,0.002225,0.931318,0.111629,0.819689,False
1,2,1.0,0.1,2,50,400,1.0,0.8,0.815343,0.002692,0.930982,0.111121,0.819861,True
2,3,0.0,0.1,2,10,400,1.0,0.8,0.815101,0.002462,0.931318,0.111629,0.819689,False



Saved xgb_spacy.json, xgb_electra.json, xgb_tfidf.json, tfidf_vectorizer.joblib, baseline_validation_selection.csv and baseline_cv_results.json to /workspace/hybrid-ai-framework/data/processed/models


## Which spaCy features matter

XGBoost gain for each of the ten stylometric features in the spaCy-only model. This speaks to research question 1: which stylometric features are most effective at separating human from AI text. It reads only the trained model, not the test rows.

In [7]:
gain = models["spacy"].get_booster().get_score(importance_type="gain")
importance = pd.DataFrame(
    {
        "feature": SPACY_FEATURE_NAMES,
        "gain": [gain.get(f"f{i}", 0.0) for i in range(len(SPACY_FEATURE_NAMES))],
    }
)
total_gain = importance["gain"].sum()
importance["share"] = importance["gain"] / total_gain if total_gain > 0 else 0.0
importance = importance.sort_values("gain", ascending=False).reset_index(drop=True)
display(importance)
importance.to_csv(MODELS_DIR / "spacy_feature_importance.csv", index=False)
print("Wrote", MODELS_DIR / "spacy_feature_importance.csv")

,feature,gain,share
0,avg_dependency_length,786.488220,0.351175
1,avg_word_length,368.329437,0.164463
2,pos_adv_ratio,332.590698,0.148505
3,ttr,177.554520,0.079280
4,pos_verb_ratio,156.656418,0.069949
5,pos_noun_ratio,143.688263,0.064158
6,pos_adj_ratio,97.434059,0.043505
7,hapax_legomena_ratio,64.130394,0.028635
8,function_word_ratio,58.866810,0.026285
9,mean_parse_tree_depth,53.852020,0.024045


Wrote /workspace/hybrid-ai-framework/data/processed/models/spacy_feature_importance.csv
